In [1]:
import os
import pandas as pd

uxm_results_path = "/data/dmytro/cfSortData/UXM_results"
import numpy as np

In [2]:
metadata = pd.read_csv(
    "/data/dmytro/cfSortData/GSE233417_samples/GSE233417_samples.csv"
)

In [ ]:
from syto.deconvolution.linear_calibrator import LinearCalibrator
from syto.deconvolution.vector_scaling_calibrator import (
    VectorScalingCalibrator,
    VectorScalingCalibratorCV,
)

calibrator = LinearCalibrator.load(
    "/home/luna.kuleuven.be/u0169940/Repos/syto/Experiments/pseudobulk/uxm_results_pseudobulk/uxm_linear_calibrator.npz"
)

In [22]:
np.sum(calibrator_vector.predict(np.reshape(sub[[sub.columns[1]]].to_numpy(), (1, 39))))

np.float64(1.0)

In [39]:
results_top25 = []
results_top25gsc = []
results_top250 = []
for file in os.listdir(uxm_results_path):
    sub = pd.read_csv(os.path.join(uxm_results_path, file))
    if "Atlas.U250" in file:
        sub.columns = ["CellType", "UXM_U250"]
    elif "U25gscSorted" in file:
        sub.columns = ["CellType", "UXM_U25_GSC_Sorted"]
    else:
        sub.columns = ["CellType", "UXM_U25"]
        sub["UXM_U25_clip0"] = calibrator_linear.predict(
            np.reshape(sub[[sub.columns[1]]].to_numpy(), (1, 39)),
            norm_method="clip0-normalize",
        )[0][0]
        sub["UXM_U25_simplex"] = calibrator_linear.predict(
            np.reshape(sub[[sub.columns[1]]].to_numpy(), (1, 39)),
            norm_method="simplex-projection",
        )[0][0]
        sub["UXM_U25_vector"] = calibrator_vector.predict(
            np.reshape(sub[[sub.columns[1]]].to_numpy(), (1, 39))
        )[0]
    sub["file"] = file.split("_")[0]
    sub["Biosample organism"] = "Homo sapiens"
    sub["Biosample type"] = "tissue"
    sub["Biosample term id"] = "not mapped"
    sub_meta = metadata[metadata["sample_geo_accession"] == file.split("_")[0]]
    if len(sub_meta) != 1:
        raise ValueError(f"Corrupted metadata for {file}")
    tissue = sub_meta["tissue"].to_list()[0]
    sub["Biosample term name"] = sub_meta["tissue"].to_list()[0]
    if "Atlas.U250" in file:
        results_top250.append(sub)
    elif "U25gscSorted" in file:
        results_top25gsc.append(sub)
    else:
        results_top25.append(sub)

In [40]:
results_top25_pd = pd.concat(results_top25, axis=0).reset_index()
results_top250_pd = pd.concat(results_top250, axis=0).reset_index()
results_top25gsc = pd.concat(results_top25gsc, axis=0).reset_index()

In [41]:
results_uxm_all = pd.merge(
    pd.merge(
        results_top25_pd,
        results_top250_pd[["CellType", "file", "UXM_U250"]],
        on=["CellType", "file"],
    ),
    results_top25gsc[["CellType", "file", "UXM_U25_GSC_Sorted"]],
    on=["CellType", "file"],
)

In [42]:
results_uxm_all.drop("index", axis=1, inplace=True)

In [45]:
results_uxm_all.to_csv("cfSort_rrbs_uxm_results_final.csv", index=False)

### Load other results

In [1]:
# target_path = "/home/luna.kuleuven.be/u0169940/Repos/syto/Experiments/RRBS/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_d041_postfiltered_min_length_50_soft_labels_pooled_jakkard/pseudobulk/deconvolutions"
# target_path = "/data/dmytro/syto_models/archive/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels/deconvolutions"
target_path = "/home/luna.kuleuven.be/u0169940/Repos/syto/Experiments/RRBS/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_d041_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk/deconvolutions"

In [2]:
file_to_model_dict = {
    "deconvolution_xgboost.csv": "XGB",
    "deconvolution_xgboost_callibrated.csv": "XGB_LCal",
    "deconvolution_3Layer_MLP.csv": "MLP",
    "deconvolution_3Layer_MLP_callibrated.csv": "MLP_LCal",
    "deconvolution_Shallow_Wide_Network.csv": "SWN",
    "deconvolution_Shallow_Wide_Network_callibrated.csv": "SWN_LCal",
    "deconvolution_nnls.csv": "NNLS",
    "deconvolution_nnls_callibrated.csv": "NNLS_LCal",
    "deconvolution_psls.csv": "PSLS",
    "deconvolution_psls_callibrated.csv": "PSLS_LCal",
}

In [8]:
results_methylbert_soft = []
for folder in os.listdir(target_path):
    sub = pd.DataFrame()
    for key, value in file_to_model_dict.items():
        partial_sub = pd.read_csv(os.path.join(target_path, folder, key))
        if not len(sub):
            sub = partial_sub
            sub.columns = ["CellType", value]
        else:
            sub[value] = partial_sub[partial_sub.columns[1]]
    sub["file"] = folder.split("_")[0]
    sub["Biosample organism"] = "Homo sapiens"
    sub["Biosample type"] = "tissue"
    sub["Biosample term id"] = "not mapped"
    sub_meta = metadata[metadata["sample_geo_accession"] == folder.split("_")[0]]
    if len(sub_meta) != 1:
        raise ValueError(f"Corrupted metadata for {folder}")
    tissue = sub_meta["tissue"].to_list()[0]
    sub["Biosample term name"] = sub_meta["tissue"].to_list()[0]
    sub["AuditGood"] = True
    sub["CellTypeProxy"] = [cfsort_tissue_to_celltypes[tissue]] * 39
    sub["TagFiltered"] = True
    results_methylbert_soft.append(sub)

In [9]:
results_methylbert_soft = pd.concat(results_methylbert_soft, axis=0).reset_index()

In [10]:
results_methylbert_soft["classifier"] = "MethylBERT"
results_methylbert_soft["labeling_scheme"] = "Hard Labels"

In [11]:
results_methylbert_soft.to_csv("cfSort_rrbs_methylbert_hard_results.csv", index=False)

### Load other results V2

In [3]:
# target_path = "/mnt/data/syto_experiments/mlflow/148674369126840102/e0b049fd91554cd7b17c08ff598af65f/artifacts/cfsort_results/top_156_features"
target_path = "/mnt/data/cfsort/deconvolution_results/U25.l4.hg19/uxm/"

In [4]:
results_soft_prior_blending = []
for folder in os.listdir(target_path):
    sub = pd.read_csv(os.path.join(target_path, folder, "deconvolution_results.csv"))
    sub["file"] = folder.split("_")[0]
    sub["Biosample organism"] = "Homo sapiens"
    sub["Biosample type"] = "tissue"
    sub_meta = metadata[metadata["sample_geo_accession"] == folder.split("_")[0]]
    if len(sub_meta) != 1:
        raise ValueError(f"Corrupted metadata for {folder}")
    tissue = sub_meta["tissue"].to_list()[0]
    sub["Biosample term name"] = sub_meta["tissue"].to_list()[0]
    results_soft_prior_blending.append(sub)

results_soft_prior_blending = pd.concat(
    results_soft_prior_blending, axis=0
).reset_index()

In [5]:
results_soft_prior_blending = results_soft_prior_blending[
    results_soft_prior_blending.columns[2:]
]

In [6]:
results_soft_prior_blending

,CellType,Classifier,Deconvolver,Calibrator,PredictedProportion,file,Biosample organism,Biosample type,Biosample term name
0,Adipocytes,methylbert,uxm,NaN,0.0249,GSM7427477,Homo sapiens,tissue,blood vessel
1,Bladder-Ep,methylbert,uxm,NaN,0.0012,GSM7427477,Homo sapiens,tissue,blood vessel
2,Blood-B,methylbert,uxm,NaN,0.0000,GSM7427477,Homo sapiens,tissue,blood vessel
3,Blood-Granul,methylbert,uxm,NaN,0.0030,GSM7427477,Homo sapiens,tissue,blood vessel
4,Blood-Mono+Macro,methylbert,uxm,NaN,0.1006,GSM7427477,Homo sapiens,tissue,blood vessel
...,...,...,...,...,...,...,...,...,...
20314,Prostate-Ep,methylbert,uxm,NaN,0.0000,GSM7427399,Homo sapiens,tissue,ovary
20315,Skeletal-Musc,methylbert,uxm,NaN,0.0124,GSM7427399,Homo sapiens,tissue,ovary
20316,Small-Int-Ep,methylbert,uxm,NaN,0.0000,GSM7427399,Homo sapiens,tissue,ovary
20317,Smooth-Musc,methylbert,uxm,NaN,0.0728,GSM7427399,Homo sapiens,tissue,ovary


In [7]:
results_soft_prior_blending["Classifier"] = None

In [8]:
results_soft_prior_blending.to_csv(
    "/home/luna.kuleuven.be/u0169940/Repos/syto/EDA/cfsort_results_v2/UXM/uxm_ported_cfsort_rrbs_results.csv",
    index=False,
)

In [3]:
import pandas as pd
all_ranks = pd.read_csv("pseudobulk_vs_tcs_ranks.csv")

In [7]:
all_ranks.sort_values("avg_rank").to_csv("pseudobulk_vs_tcs_ranks_sorted.tsv", sep="\t")

In [10]:
all_ranks.sort_values("avg_rank").loc[0:50]["Classifier"].value_counts()

Classifier
dismir            20
methylbert        15
cancerdetector    15
Name: count, dtype: int64

In [11]:
all_ranks.columns

Index(['Classifier', 'Labeling Scheme', 'Prior', 'Feature Scheme',
       'Deconvolver', 'Calibrator', 'method', 'method_key', 'pseudobulk_r2',
       'tcs', 'tcs_per_sample', 'rank_pseudobulk', 'rank_tcs', 'avg_rank',
       'rank_r2', 'rank_cosine_sim', 'rank_mse', 'rank_mae', 'rank_kl',
       'rank_max_error', 'rank_loa_width', 'rank_worst_class_loa_width'],
      dtype='object')

In [14]:
all_ranks.sort_values("avg_rank").loc[0:50]["Calibrator"].value_counts()

Calibrator
linear_simplex_projection    20
vector_scaling               14
linear_clip_normalize        12
uncalibrated                  4
Name: count, dtype: int64